# LoRA fine tuning to improve pubmedbert performance on exaggeration task

In [7]:
import sys
from pyprojroot import here

sys.path.insert(0, str(here()))

import yaml
with open(here("configs/base.yaml"), "r") as f:
    base_config = yaml.safe_load(f)

from src.data_holdout import load_hf_dataset, process_and_pool_data, get_fold_from_disk

Loading dataset, copenlu/scientific-exaggeration-detection
Dataset({
    features: ['original_file_id', 'press_release_conclusion', 'press_release_strength', 'abstract_conclusion', 'abstract_strength', 'exaggeration_label'],
    num_rows: 100
})
Created held-out split: train_dev=530 test=133
int64
[2 0 1]
Applying split of: 5
Fold 1: train=424 val=106
Fold 2: train=424 val=106
Fold 3: train=424 val=106
Fold 4: train=424 val=106
Fold 5: train=424 val=106

==================== SANITY CHECK: KFOLD SPLITS ====================
Total samples: 663 | Num folds: 5
label dtype: int64
unique labels: [0, 1, 2]

Overall label counts:
exaggeration_label
same           406
exaggerates    144
downplays      113
Name: count, dtype: int64

Overall label %:
exaggeration_label
same           61.24
exaggerates    21.72
downplays      17.04
Name: proportion, dtype: float64

Fold 1
  Train size=424 | Val size=106
  Train label %:
exaggeration_label
downplays      16.98
exaggerates    21.70
same           61.

In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report
from peft import LoraConfig, get_peft_model, TaskType

import numpy as np

/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load Pubmedbert model and tokenizer

In [3]:
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

### Create tokenize_fold and compute_metrics helper functions

In [4]:
def tokenize_fold(df, max_length=256):
    ds = Dataset.from_pandas(df, preserve_index=False)

    def _tokenize(examples):
        return tokenizer(
            examples["abstract_conclusion"],
            examples["press_release_conclusion"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    cols_to_remove = [c for c in ds.column_names if c != "exaggeration_label"]
    ds = ds.map(_tokenize, batched=True, remove_columns=cols_to_remove)
    ds = ds.rename_column("exaggeration_label", "labels")
    ds.set_format("torch")
    return ds

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    return {"macro_f1": macro_f1}

## Test out baseline for LoRA before optimizing hyperparameters

### Create LoRA model

In [13]:

def make_lora_model():
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
    )

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        target_modules=["query", "value"],
    )

    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    return model



### Define training arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="outputs/pubmedbert_lora_fold0",
    eval_strategy=base_config["eval_strategy"],
    save_strategy=base_config["save_strategy"],
    load_best_model_at_end=base_config["load_best_model_at_end"],
    metric_for_best_model=base_config["metric_for_best_model"],
    greater_is_better=base_config["greater_is_better"],
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    num_train_epochs=5,
    warmup_ratio=0.1,
    fp16=base_config["fp16"],
    seed=base_config["seed"],
    report_to=base_config["report_to"],
    optim=base_config["optim"],
)


### Load and tokenize the data

In [9]:
full_data = load_hf_dataset("copenlu/scientific-exaggeration-detection")
full_df = process_and_pool_data(full_data["train"], full_data["test"])

train_fold, val_fold = get_fold_from_disk(
    full_df,
    fold=0,
    k=base_config["cv_k"],
    seed=base_config["cv_seed"],
)

train_ds = tokenize_fold(train_fold, max_length=base_config["max_length"])
val_ds = tokenize_fold(val_fold, max_length=base_config["max_length"])

Map: 100%|██████████| 106/106 [00:00<00:00, 12138.81 examples/s]


### Train LoRA model on one fold

In [14]:
model = make_lora_model()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=base_config["early_stopping_patience"]
    )],
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1570.45it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | 

trainable params: 297,219 || all params: 109,781,766 || trainable%: 0.2707


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.994758,0.253411
2,No log,1.003465,0.253411
3,No log,0.969788,0.253411
4,No log,0.980054,0.252465


/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(

{'eval_loss': 0.9947576522827148, 'eval_macro_f1': 0.253411306042885, 'eval_runtime': 10.4782, 'eval_samples_per_second': 10.116, 'eval_steps_per_second': 1.336, 'epoch': 4.0}


### Run baseline LoRA model on all 5 folds

In [17]:
# Tokenize all folds
full_data = load_hf_dataset("copenlu/scientific-exaggeration-detection")
full_df = process_and_pool_data(full_data["train"], full_data["test"])
full_dataset = Dataset.from_pandas(full_df, preserve_index=False)

def _tokenize(examples):
    return tokenizer(
        examples["abstract_conclusion"],
        examples["press_release_conclusion"],
        truncation=True,
        padding="max_length",
        max_length=base_config["max_length"],
    )

full_dataset = full_dataset.map(_tokenize, batched=True)

full_dataset = full_dataset.rename_column("exaggeration_label", "labels")
full_dataset.set_format("torch")

Map: 100%|██████████| 663/663 [00:00<00:00, 11115.20 examples/s]


In [18]:
# get folds
train_folds, val_folds = get_fold_from_disk(
    full_df,
    k=base_config["cv_k"],
    seed=base_config["cv_seed"],
)

fold_scores = []

for fold in range(base_config["cv_k"]):

    train_indices = train_folds[fold].index.tolist() #get fold indices
    val_indices = val_folds[fold].index.tolist()

    train_ds = full_dataset.select(train_indices) #subset tokenized dataset based on fold indices
    val_ds = full_dataset.select(val_indices)

    training_args = TrainingArguments(
        output_dir=f"outputs/pubmedbert_lora_fold{fold}", #save each output in new folder based on fold
        eval_strategy=base_config["eval_strategy"],
        save_strategy=base_config["save_strategy"],
        load_best_model_at_end=base_config["load_best_model_at_end"],
        metric_for_best_model=base_config["metric_for_best_model"],
        greater_is_better=base_config["greater_is_better"],
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-4,
        weight_decay=0.01,
        num_train_epochs=5,
        warmup_ratio=0.1,
        fp16=base_config["fp16"],
        seed=base_config["seed"],
        report_to=base_config["report_to"],
        optim=base_config["optim"],
)

    model = make_lora_model()

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=base_config["early_stopping_patience"])],
    )

    trainer.train()
    metrics = trainer.evaluate()

    fold_f1 = metrics["eval_macro_f1"]
    fold_scores.append(fold_f1)

    print(f"Fold {fold}: {fold_f1:.4f}")

print(f"\nPubMedBERT + LoRA: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2996.15it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight   

trainable params: 297,219 || all params: 109,781,766 || trainable%: 0.2707


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.988736,0.253411
2,No log,0.994393,0.253411
3,No log,0.961885,0.254902
4,No log,0.972506,0.252465
5,No log,0.973897,0.250000


/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2941.02it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | 

trainable params: 297,219 || all params: 109,781,766 || trainable%: 0.2707


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.939636,0.250980
2,No log,0.929489,0.253411
3,No log,0.920874,0.250980
4,No log,0.917593,0.253411
5,No log,0.918840,0.253411


/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(

Fold 4: 0.2534

PubMedBERT + LoRA: 0.2537 ± 0.0006


In [19]:
model.print_trainable_parameters()

trainable params: 297,219 || all params: 109,781,766 || trainable%: 0.2707


In [20]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
)

for name, _ in base_model.named_modules():
    if "query" in name or "value" in name or "key" in name:
        print(name)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4062.09it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | 

bert.encoder.layer.0.attention.self.query
bert.encoder.layer.0.attention.self.key
bert.encoder.layer.0.attention.self.value
bert.encoder.layer.1.attention.self.query
bert.encoder.layer.1.attention.self.key
bert.encoder.layer.1.attention.self.value
bert.encoder.layer.2.attention.self.query
bert.encoder.layer.2.attention.self.key
bert.encoder.layer.2.attention.self.value
bert.encoder.layer.3.attention.self.query
bert.encoder.layer.3.attention.self.key
bert.encoder.layer.3.attention.self.value
bert.encoder.layer.4.attention.self.query
bert.encoder.layer.4.attention.self.key
bert.encoder.layer.4.attention.self.value
bert.encoder.layer.5.attention.self.query
bert.encoder.layer.5.attention.self.key
bert.encoder.layer.5.attention.self.value
bert.encoder.layer.6.attention.self.query
bert.encoder.layer.6.attention.self.key
bert.encoder.layer.6.attention.self.value
bert.encoder.layer.7.attention.self.query
bert.encoder.layer.7.attention.self.key
bert.encoder.layer.7.attention.self.value
bert.enc